# vLLM& GKE Production

**Module 13 · Lesson 13.6**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Why vLLM: Continuous Batching
- PagedAttention: The KV Cache, Paged
- The Batching Knob: Throughput vs Latency
- The vLLM Server + Tensor Parallelism
- Deploy On GKE
- Autoscale On The Right Signal

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install openai -q

# Reproducibility - the lesson uses seeded random values throughout

## Ollama serves one at a time; production needs a fleet

> 💡 **Info**
>
> In Lesson 13.5 you ran an open model on your own hardware with Ollama, and you saw its ceiling: it serves essentially **one request at a time**. That is perfect for development and a low-traffic tool, but production is a different animal — **hundreds of concurrent users**, **zero-downtime deploys**, and a serving layer that **autoscales and heals itself**. To get there you graduate to **vLLM on GKE**. vLLM is the standard open-model serving engine, and it earns its throughput with two ideas. **Continuous batching** reschedules the GPU’s batch on *every forward pass*: the instant one request finishes, its slot is refilled from the queue, so the GPU never sits idle waiting for the slowest request in the batch. **PagedAttention** stores the KV cache in fixed, non-contiguous blocks — like the pages of operating-system virtual memory — so it stops wasting VRAM on reservations and packs far more sequences onto the card. Together they give roughly **ten to twenty times** the throughput of naive serving, and that throughput is precisely what makes a self-hosted GPU *cheap* per token. **GKE** then runs vLLM as a fleet: a Deployment of GPU pods, autoscaled on the one signal that actually moves under load (the request queue, not CPU), behind health checks and rolling updates. This lesson walks the whole stack — the two engines, the batching knob, multi-GPU serving, the GKE deployment, autoscaling done right, and the economics that decide whether to run it at all.

### 🎯 What you will be able to do after this lesson

- **Build** production-serving tooling — a static-vs-continuous batching model, a PagedAttention VRAM-waste model, a max-num-seqs throughput/latency sweep, a tensor-parallel sizer, a GKE cold-start model, a right-signal autoscaler, and a cost-per-token calculator — as runnable models.

- **Compare** static with continuous batching, naive KV reservation with PagedAttention, and a CPU HPA with a queue-depth HPA.

- **Operate** `vllm serve` with `--tensor-parallel-size` / `--max-num-seqs`, a GKE Deployment with a GPU node pool, and an HPA on the request queue.

- **Evaluate** a production serving decision: at your concurrency and utilization, is vLLM on GKE cheaper than a managed API or Ollama?

> 📦 **Info**
>
> ✅ Before you startIn **13.5** Ollama served one request at a time; vLLM is the graduation the moment you need real concurrency. In **13.4** the KV cache grew with context and ate VRAM — PagedAttention is how vLLM packs it, and it is the memory that continuous batching needs to keep the batch full. And in **12.7** you scaled on concurrency, not CPU; here it is more extreme still, because vLLM pre-allocates its VRAM, so the only signal that moves is the queue. Serving as an operational discipline comes next, in Module 14.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🍽️ **Analogy**
>
> vLLM versus naive serving is **a well-run restaurant versus one that seats by the whole room**. The bad restaurant seats everyone at once and will not seat the next group until the *last* table of the current seating leaves — so tables sit empty while one slow party lingers over dessert. The good restaurant seats the next party the instant a table frees up, keeping every table full all night (that is **continuous batching**). And it does not reserve a whole section per booking — it seats parties at whatever tables are open, packing the room tight (that is **PagedAttention**). Same dining room, far more meals served. **Where it breaks down:** a restaurant can add tables, but a GPU cannot — so the whole art is keeping the tables you have full, which is what these two engines do.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“More replicas is how you get throughput.”** A single vLLM GPU with continuous batching already serves hundreds of concurrent requests; you add replicas for capacity and high availability, not to fix a per-GPU throughput problem.
> - **“Autoscale on CPU or GPU memory.”** Both stay flat, because vLLM pre-allocates its VRAM at startup. The only signal that moves under load is the request queue, so that is what you scale on.

> 💡 **Info**
>
> ⚠️ Anti-patternTwo ways to run a broken production stack. **Scaling a GPU-bound LLM on CPU utilization** — the classic broken autoscaler that never fires, because a vLLM pod’s CPU does not rise with load. And **scaling to zero with a slow cold start** — a traffic burst hits a pod that needs minutes to download and load the model, so the burst is dropped; keep min replicas warm and cache the weights on a volume.

---

## 🎯 Concept 1: Why vLLM: Continuous Batching

### Why vLLM: Continuous Batching

Naive serving uses static batching - it launches a fixed batch and waits for the longest request to finish, so short requests hold their GPU slot idle (60-80 percent of GPU time wasted on padding). Continuous batching reschedules every forward pass: a finished slot is refilled from the queue instantly, so the GPU never idles - up to ten to twenty times the throughput.

Start with the engine that makes vLLM fast. Naive serving uses **static batching**: it gathers a fixed group of requests, launches them together, and waits for the *longest* one to finish before it can start the next batch. But requests generate wildly different numbers of tokens, so the short ones sit in their GPU slot doing nothing while the long one grinds on — on real workloads, **sixty to eighty percent** of the GPU’s time is wasted on this idle padding. **Continuous batching** (vLLM’s scheduler) fixes it by rescheduling the batch on *every single forward pass*: the instant a request finishes, its slot is **evicted and refilled** from the waiting queue, without waiting for a batch boundary. The GPU stays full, so throughput jumps — up to **ten to twenty times** naive serving, and it *lowers* p50 latency at the same time, because requests are not stuck waiting for a batch to form. This is the single biggest win in LLM serving, and it is the reason vLLM exists. The block prices the waste, keyless.

> 🥒 **Analogy**
>
> It is **seating the next party the moment a table frees up**. A restaurant that only re-seats when the entire room turns over leaves tables empty while one slow party lingers — that is static batching, holding every GPU slot until the longest request finishes. A restaurant that seats a waiting party the instant any table opens keeps every table working all night — that is continuous batching, refilling each GPU slot the moment a request completes. Same room, same tables, far more diners served, and nobody waits at the door for a whole seating to clear.

A static batch of four requests generates 64, 96, 128, and 512 tokens. The GPU holds all 4 slots until the 512-token request finishes. Roughly how much of that GPU time is useful work?

**📝 `01_continuous_batching.py`** - *runnable*

In [ ]:
# STATIC batching launches a fixed batch together and waits for the LONGEST to finish - the short
# requests hold their GPU slot idle. CONTINUOUS batching (vLLM) reschedules EVERY forward pass: a finished
# request is evicted immediately and a queued one takes its slot, so the GPU never idles. (Illustrative.)
outputs = [64, 96, 128, 512]           # output tokens each of 4 batched requests generates
batch, longest = len(outputs), max(outputs)
static_slotsteps = batch * longest     # static reserves batch x longest slot-steps (all wait for the longest)
useful = sum(outputs)                  # the tokens actually generated
static_waste = 1 - useful / static_slotsteps
print("A batch of {} requests generating {} tokens (longest {}):".format(batch, outputs, longest))
print("  STATIC: reserves {} slot-steps (all {} slots held until the {}-token request finishes)".format(static_slotsteps, batch, longest))
print("  useful work: {} tokens -> {:.0%} of the GPU-time is WASTED padding (idle slots).".format(useful, static_waste))
print()
print("  CONTINUOUS (vLLM): a finished slot is refilled from the queue the same step, so slots stay busy.")
print("  With a full queue, that recovers the wasted {:.0%} - about {:.1f}x the throughput on this batch,".format(static_waste, static_slotsteps / useful))
print("  and on real deep-queue workloads continuous batching reaches 10-23x while ALSO cutting p50 latency.")
print("Continuous batching is the single biggest serving win - it is why vLLM exists.")

# Output:
# A batch of 4 requests generating [64, 96, 128, 512] tokens (longest 512):
#   STATIC: reserves 2048 slot-steps (all 4 slots held until the 512-token request finishes)
#   useful work: 800 tokens -> 61% of the GPU-time is WASTED padding (idle slots).
#
#   CONTINUOUS (vLLM): a finished slot is refilled from the queue the same step, so slots stay busy.
#   With a full queue, that recovers the wasted 61% - about 2.6x the throughput on this batch,
#   and on real deep-queue workloads continuous batching reaches 10-23x while ALSO cutting p50 latency.
# Continuous batching is the single biggest serving win - it is why vLLM exists.

- **Static batching** reserves the whole batch until the longest request finishes, so the short ones hold idle slots.
- On this batch about **sixty percent** of the GPU-time is wasted padding — and real workloads run sixty to eighty percent.
- **Continuous batching** refills a finished slot from the queue the same step, so the GPU stays full — recovering that waste.
- On deep-queue production traffic that reaches ten to twenty-plus times the throughput, while also cutting p50 latency.

#### 💡 What just happened

⚡ What just happened?Static batching wastes 60-80 percent of GPU time holding idle slots until the longest request finishes; continuous batching refills a finished slot from the queue every forward pass, so the GPU never idles - ~10-20x throughput and lower p50. Tradeoff / the framing: this is a pure scheduling win, the reason vLLM exists. Next: the memory engine that keeps the batch full - PagedAttention.

- A batch of requests with different output lengths; toggle static (slots held to the longest, padding shaded) vs continuous
- A finished slot is instantly refilled from a queue; a throughput and waste readout updates

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: PagedAttention: The KV Cache, Paged

### PagedAttention: The KV Cache, Paged

Naive serving reserves a contiguous block of the full max context per request, wasting VRAM on tokens it never generates. PagedAttention stores the KV cache in fixed non-contiguous blocks, exactly like OS virtual-memory pages, so waste is at most one block - packing far more sequences into the same GPU. When it fills, vLLM preempts an older sequence.

Continuous batching needs a full batch to work, and the thing that limits how many sequences fit on a GPU is the **KV cache** (the attention state from 13.4). Naive serving reserves a **contiguous** block of VRAM sized to the *maximum* context length for *every* request — even one that ends up generating a handful of tokens — so most of that reservation is wasted, and the leftover gaps fragment the memory. **PagedAttention** is vLLM’s fix, and it borrows the oldest trick in operating systems: **paging**. It stores the KV cache in **fixed, non-contiguous blocks** (typically sixteen tokens each), just like OS virtual-memory pages, allocating a new page only when a sequence actually needs it. Waste drops to at most one partial block per request — roughly **half the VRAM** that naive serving throws away — so many more sequences fit on the same card, which means a fuller batch, which means more of the throughput continuous batching can deliver. And when the KV cache does fill up, vLLM **preempts** (evicts) an older sequence to make room, which is why `gpu_cache_usage_perc` is a metric to watch. The block prices the waste, keyless.

> 🚗 **Analogy**
>
> It is **a parking garage with numbered spots instead of reserved floors**. The naive design gives every arriving car its own *entire floor*, in case it brought a bus — so a compact car wastes a whole floor, and the garage fills after a few cars. PagedAttention parks each car in the next open numbered spot, wherever it is, spread across floors; a car takes exactly the spots it needs and no more. The garage packs in many times more cars for the same size — and that is the KV cache: fixed small spots (pages), allocated on demand, non-contiguous, no reserved floors going to waste.

PagedAttention stores the KV cache in fixed 16-token blocks instead of one contiguous max-length reservation per request. What does it mainly buy you?

**📝 `02_paged_attention.py`** - *runnable*

In [ ]:
# PagedAttention: the KV cache is the other bottleneck. Naive serving reserves a CONTIGUOUS block of
# max_model_len per request (for tokens it may never generate) -> huge waste + fragmentation. PagedAttention
# stores the KV cache in fixed BLOCKS (like OS virtual-memory pages), so waste is at most one block.
max_len, block = 4096, 16               # max_model_len tokens; a KV page is 16 tokens
actual = [200, 512, 1500]               # the actual context lengths of 3 live requests
used = sum(actual)
naive_reserved = len(actual) * max_len                                  # each reserves the full max_len
paged_reserved = sum((a + block - 1) // block * block for a in actual)  # each reserves ceil(len/block) pages
print("Three requests with contexts {} (max_model_len {}):".format(actual, max_len))
print("  naive (contiguous max reserve): {:>6} tokens reserved, {} used -> {:.0%} wasted".format(naive_reserved, used, 1 - used / naive_reserved))
print("  PagedAttention (16-token pages): {:>6} tokens reserved, {} used -> {:.0%} wasted".format(paged_reserved, used, 1 - used / paged_reserved))
print()
seqs_naive = naive_reserved // max_len          # in a fixed KV budget, naive fits few (each hogs max_len)
mult = naive_reserved / paged_reserved
print("In the SAME VRAM, PagedAttention fits about {:.1f}x more concurrent sequences (it packs the KV cache tight).".format(mult))
print("More concurrent sequences = a fuller batch = more of the throughput continuous batching can deliver.")
print("When the KV cache fills, vLLM PREEMPTS (evicts) an older sequence to make room - watch gpu_cache_usage_perc.")

# Output:
# Three requests with contexts [200, 512, 1500] (max_model_len 4096):
#   naive (contiguous max reserve):  12288 tokens reserved, 2212 used -> 82% wasted
#   PagedAttention (16-token pages):   2224 tokens reserved, 2212 used -> 1% wasted
#
# In the SAME VRAM, PagedAttention fits about 5.5x more concurrent sequences (it packs the KV cache tight).
# More concurrent sequences = a fuller batch = more of the throughput continuous batching can deliver.
# When the KV cache fills, vLLM PREEMPTS (evicts) an older sequence to make room - watch gpu_cache_usage_perc.

- **Naive** reserves the full `max_model_len` per request, so most of the reservation is unused — the wasted share is large.
- **PagedAttention** reserves only the pages a request actually uses, so waste is at most one block — a tiny fraction.
- In the same VRAM that packs **several times more concurrent sequences**, which is exactly what a full continuous batch needs.
- When the cache fills, vLLM **preempts** an older sequence — watch `gpu_cache_usage_perc` so you scale before that bites.

#### 💡 What just happened

⚡ What just happened?PagedAttention stores the KV cache in fixed non-contiguous pages (like OS virtual memory), cutting the wasted VRAM of a contiguous max-length reservation by roughly half and fitting several times more sequences - the fuel for continuous batching. Tradeoff: a little page-table bookkeeping, in exchange for a far fuller GPU. Next: the knob that sets how full you push the batch.

- A VRAM strip filled by three requests; toggle naive (each reserves the full max_model_len, grey waste) vs paged (tight fixed blocks)
- The sequences-that-fit count jumps as the waste collapses

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: The Batching Knob: Throughput vs Latency

### The Batching Knob: Throughput vs Latency

--max-num-seqs sets how many sequences run at once. A bigger batch pushes aggregate throughput toward the GPU ceiling, but the sequences share the GPU, so each request’s own token rate falls. The knee is where throughput saturates. Chunked prefill interleaves a long prompt’s prefill with ongoing decodes so it does not block everyone.

Continuous batching gives you a dial: **`--max-num-seqs`**, the maximum number of sequences the server runs concurrently. It is a genuine tradeoff. A **bigger batch** pushes *aggregate* throughput up toward the GPU’s ceiling — more total tokens per second across all users. But those sequences **share one GPU**, so as the batch grows, *each individual request’s* token rate **falls**: high total throughput, slower per-user streaming. The sweet spot is the **knee**, where aggregate throughput saturates the GPU; past it you add per-request latency for almost no extra throughput. So you set `max-num-seqs` to your traffic shape — small for latency-sensitive interactive chat, large for throughput-oriented batch jobs. One more piece: a long prompt’s **prefill** can hog the GPU and stall everyone else (head-of-line blocking), so vLLM uses **chunked prefill** — it splits a long prefill into fixed chunks and interleaves them with ongoing decodes, so one giant prompt does not freeze the stream for everybody. The block sweeps the knob, keyless.

> 🚌 **Analogy**
>
> It is **the size of the bus**. A taxi (a tiny batch) gets its one rider there fast, but it moves very few people per hour. A big bus (a large batch) moves far more people per hour — higher throughput — but each rider’s trip is slower because of all the stops. There is a right size for your route: a shuttle van for a quiet street, a full coach for a packed commuter line. `max-num-seqs` is that bus size — you pick it for whether you are optimizing each rider’s speed (low latency) or total riders moved (high throughput).

You raise --max-num-seqs from 8 to 128 on a busy server. What happens to aggregate throughput and to each request’s own token rate?

**📝 `03_batching_knob.py`** - *runnable*

In [ ]:
# The knob: --max-num-seqs sets how many sequences run at once. A bigger batch pushes AGGREGATE throughput
# toward the GPU ceiling, but the sequences share the GPU, so each request's own token rate FALLS. Find the knee.
GPU_CEILING = 2400          # aggregate decode tokens/s at saturation (illustrative, 70B FP8 on one H100)
PER_SEQ_MAX = 90            # a single lightly-loaded sequence decodes ~90 tok/s
print("max-num-seqs  aggregate tok/s   per-request tok/s   note")
for m in [8, 32, 128, 512]:
    agg = min(GPU_CEILING, m * PER_SEQ_MAX)     # throughput rises with batch until it hits the ceiling
    per_req = agg / m                            # but each of m sequences gets a share
    note = "under-utilised" if agg < GPU_CEILING else "saturated (the knee is ~here)"
    print("  {:>4}          {:>6}            {:>5.0f}              {}".format(m, agg, per_req, note))
print()
print("Small batch: low throughput, snappy per-request. Big batch: full GPU, slower per-request. The knee is")
print("where aggregate throughput saturates ({} tok/s here) - past it you add latency for almost no throughput.".format(GPU_CEILING))
print("Chunked prefill (default 2048-token chunks) interleaves a long prompt's prefill with ongoing decodes,")
print("so one big prompt does not block everyone else (head-of-line blocking). Set max-num-seqs to your traffic.")

# Output:
# max-num-seqs  aggregate tok/s   per-request tok/s   note
#      8             720               90              under-utilised
#     32            2400               75              saturated (the knee is ~here)
#    128            2400               19              saturated (the knee is ~here)
#    512            2400                5              saturated (the knee is ~here)
#
# Small batch: low throughput, snappy per-request. Big batch: full GPU, slower per-request. The knee is
# where aggregate throughput saturates (2400 tok/s here) - past it you add latency for almost no throughput.
# Chunked prefill (default 2048-token chunks) interleaves a long prompt's prefill with ongoing decodes,
# so one big prompt does not block everyone else (head-of-line blocking). Set max-num-seqs to your traffic.

- As `max-num-seqs` grows, **aggregate throughput** climbs toward the GPU ceiling — more total tokens per second.
- But **per-request token rate** falls, because the sequences share one GPU — higher latency per user.
- The **knee** is where aggregate saturates; past it you add latency for almost no extra throughput.
- **Chunked prefill** interleaves a long prompt’s prefill with decodes so one big request does not block everyone (head-of-line blocking).

#### 💡 What just happened

⚡ What just happened?--max-num-seqs trades per-request latency for aggregate throughput: a bigger batch fills the GPU but slows each request; the knee is where throughput saturates. Chunked prefill keeps one long prompt from blocking the batch. Tradeoff: set the knob for interactive (small) vs batch (large) traffic. Next: serving a model too big for one GPU.

- A max-num-seqs slider; aggregate throughput climbs to a GPU-ceiling plateau
- A per-request token-rate bar falls as the batch grows; the knee is marked

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: The vLLM Server + Tensor Parallelism

### The vLLM Server + Tensor Parallelism

`vllm serve ` starts an OpenAI-compatible server on port 8000 - the same /v1/chat/completions your app calls. A model too big for one GPU is split across GPUs with --tensor-parallel-size, which shards every layer; the size is roughly the model VRAM over one GPU’s usable VRAM, rounded to a power of two. --gpu-memory-utilization sets the VRAM claimed.

Running vLLM is one command: **`vllm serve <model>`** starts an **OpenAI-compatible** server on port 8000, so your app calls the *same* `/v1/chat/completions` it already uses — just like Ollama, but built for scale. The production question is what to do when a model is **too big for one GPU**. The answer is **tensor parallelism**: `--tensor-parallel-size N` shards *every layer* of the model across N GPUs, which compute their slices in lockstep and combine the result. The size you need is roughly the **model’s VRAM divided by one GPU’s usable VRAM**, rounded up to a power of two (it has to divide the attention heads evenly). A seventy-billion model in FP16 (~140 GB) needs two eighty-gigabyte GPUs; in FP8 (~70 GB) it fits on one; a four-hundred-billion model in FP8 needs eight. Two related flags: `--gpu-memory-utilization` (how much of each card vLLM claims for weights plus KV cache, often 0.90) and `--max-model-len` (the context cap). Keep tensor parallelism *inside one node*, where the GPUs share fast NVLink; go to pipeline parallelism across nodes. The block sizes the split, keyless.

> 🎹 **Analogy**
>
> Tensor parallelism is **a piano piece too big for one player, split across several**. One pianist cannot reach a passage written across nine octaves at once — so you seat two or four players at the keyboard, each responsible for a range of keys, all reading the same score and striking their notes in perfect sync. To the audience it is one seamless performance. A model sharded with tensor parallelism is that ensemble: each GPU holds a slice of every layer, they compute together on every token, and the client sees one endpoint — but the work is spread across the players so the too-big piece can actually be played.

A 70B model in FP16 needs about 140 GB. Your GPUs are 80 GB each. What is the right vLLM setting?

**📝 `04_tensor_parallelism.py`** - *runnable*

In [ ]:
# A model too big for one GPU is split across GPUs with TENSOR PARALLELISM (--tensor-parallel-size). The
# rule of thumb: TP >= ceil(model VRAM / usable GPU VRAM), rounded up to a power of 2 (it must divide the heads).
GPU_VRAM, UTIL = 80, 0.90        # an 80 GB GPU; --gpu-memory-utilization 0.90 leaves room for the KV cache
usable = GPU_VRAM * UTIL
def tp_size(model_gb):
    need = math.ceil(model_gb / usable)
    return 1 if need <= 1 else 2 ** math.ceil(math.log2(need))
for name, gb in [("70B FP8 (~70 GB)", 70), ("70B FP16 (~140 GB)", 140), ("405B FP8 (~405 GB)", 405)]:
    tp = tp_size(gb)
    print("  {:<20} needs tensor-parallel-size {}  ({} x {:.0f} GB usable = {:.0f} GB across GPUs)".format(name, tp, tp, usable, tp * usable))
print()
print("`vllm serve <model> --tensor-parallel-size {} --gpu-memory-utilization {} --max-model-len 8192`".format(tp_size(140), UTIL))
print("gives an OpenAI-compatible endpoint on :8000 - the SAME /v1/chat/completions your app already calls.")
print("TP splits every layer across the GPUs; keep it inside one node (fast NVLink) and use pipeline parallel across nodes.")

# Output:
#   70B FP8 (~70 GB)     needs tensor-parallel-size 1  (1 x 72 GB usable = 72 GB across GPUs)
#   70B FP16 (~140 GB)   needs tensor-parallel-size 2  (2 x 72 GB usable = 144 GB across GPUs)
#   405B FP8 (~405 GB)   needs tensor-parallel-size 8  (8 x 72 GB usable = 576 GB across GPUs)
#
# `vllm serve <model> --tensor-parallel-size 2 --gpu-memory-utilization 0.9 --max-model-len 8192`
# gives an OpenAI-compatible endpoint on :8000 - the SAME /v1/chat/completions your app already calls.
# TP splits every layer across the GPUs; keep it inside one node (fast NVLink) and use pipeline parallel across nodes.

- A model bigger than one GPU is split with **tensor parallelism** — `--tensor-parallel-size` shards every layer across the GPUs.
- The size is roughly the **model VRAM over one GPU’s usable VRAM**, rounded to a power of two — here 70B FP16 needs two GPUs, 405B FP8 needs eight.
- `vllm serve` then exposes the **same OpenAI-compatible endpoint** on port 8000 that your app already calls.
- Keep tensor parallelism inside one node (fast NVLink); across nodes you switch to pipeline parallelism.

#### 💡 What just happened

⚡ What just happened?`vllm serve` gives an OpenAI-compatible server, and --tensor-parallel-size shards a too-big model across GPUs (~model VRAM / GPU VRAM, rounded to a power of two). Tradeoff: more GPUs and NVLink communication, in exchange for serving a model that does not fit one card. Next: running this as a fleet on GKE.

- A model-size slider; when the model exceeds one GPU the widget splits it across 2, 4, or 8 GPUs
- The layers stripe across the cards; the tensor-parallel-size is shown

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Deploy On GKE

### Deploy On GKE

A GKE Deployment runs N vLLM pods, each requesting a GPU on a GPU node pool. The catch is the cold start: a pod is not ready until it has downloaded and loaded the weights - minutes, not seconds. So keep min replicas warm and mount a weights-cache volume; readiness probes hit /health so a pod only takes traffic after the model loads.

Now run vLLM as a production fleet. On **GKE** you define a **Deployment** of vLLM pods, each requesting a GPU (`resources: nvidia.com/gpu: 1`), scheduled onto a **GPU node pool** you create with the accelerator type and the modern `gpu-driver-version` flag (which installs the driver for you — no manual DaemonSet). The one thing that surprises everyone is the **cold start**: a pod is *not ready* until it has **downloaded and loaded the model weights**, and for a large model that is **minutes, not seconds**. That has two consequences you must design for. First, a sudden traffic burst *cannot* be met by a fresh pod in time, so you keep a floor of **warm min replicas** (never scale to zero for a latency-sensitive service). Second, you skip the slow download by mounting a **PersistentVolumeClaim** at the weights cache, so the model is already on disk when a new pod starts. **Readiness and liveness probes** hit `/health`, so a pod only receives traffic *after* the model is loaded, and a stuck pod gets restarted. The block models the cold start, keyless.

> 👨‍🍳 **Analogy**
>
> A cold GKE pod is **a new chef who has to learn the entire menu before their first shift**. You cannot call one in off the street mid-rush and expect dishes in seconds — they need time to study the recipes and prep their station (that is the model download and load, minutes long). So a smart kitchen keeps enough trained chefs already on the line (warm min replicas) to cover a rush, and hands a new hire the recipe book on day one instead of making them reinvent it (the cached weights on a volume). The readiness probe is the head chef checking a new cook is ready before sending them any orders.

Your GKE HPA scales the vLLM Deployment to zero when idle to save money. A traffic spike arrives. What happens?

**📝 `05_gke_cold_start.py`** - *runnable*

In [ ]:
# On GKE, a Deployment runs N vLLM pods, each claiming a GPU (resources: nvidia.com/gpu: 1). The catch is
# COLD START: a pod is not ready until it has DOWNLOADED and loaded the weights - minutes, not seconds.
replicas, gpu_per_pod = 3, 1
model_gb, net_gbps, load_s = 140, 2.5, 60          # weights size; node download bandwidth; weight-load time
download_s = model_gb * 8 / net_gbps               # GB -> gigabits / (gigabits per second)
cold_start_s = download_s + load_s
print("Deployment: {} replicas x {} GPU = {} GPUs (each pod requests nvidia.com/gpu: {}).".format(replicas, gpu_per_pod, replicas * gpu_per_pod, gpu_per_pod))
print("Cold start of one pod (a {} GB model):".format(model_gb))
print("  download: {:.0f} s + load: {} s = {:.0f} s (~{:.0f} min) before the readiness probe passes.".format(download_s, load_s, cold_start_s, cold_start_s / 60))
print()
print("So a burst cannot be met by a cold pod in time. Two fixes:")
print("  1) mount a PersistentVolumeClaim at the HF cache -> the weights are already there, skip the download.")
print("  2) keep min replicas warm (never scale to zero) so there is always capacity while a new pod boots.")
print("Readiness/liveness probes hit /health; a pod only takes traffic AFTER the model is loaded.")

# Output:
# Deployment: 3 replicas x 1 GPU = 3 GPUs (each pod requests nvidia.com/gpu: 1).
# Cold start of one pod (a 140 GB model):
#   download: 448 s + load: 60 s = 508 s (~8 min) before the readiness probe passes.
#
# So a burst cannot be met by a cold pod in time. Two fixes:
#   1) mount a PersistentVolumeClaim at the HF cache -> the weights are already there, skip the download.
#   2) keep min replicas warm (never scale to zero) so there is always capacity while a new pod boots.
# Readiness/liveness probes hit /health; a pod only takes traffic AFTER the model is loaded.

- A **Deployment** runs N pods, each claiming a GPU (`nvidia.com/gpu: 1`) on a GPU node pool.
- The **cold start** is the model download plus load — minutes, not seconds — before the readiness probe passes.
- So a burst cannot be met by a cold pod; you keep **min replicas warm** and never scale a latency service to zero.
- A **cache volume** at the weights path skips the download, and `/health` probes gate traffic until the model is loaded.

#### 💡 What just happened

⚡ What just happened?A GKE Deployment runs vLLM pods that each claim a GPU, but a pod’s cold start (download + load the weights) is minutes - so keep min replicas warm, cache weights on a PVC, and gate traffic with /health probes. Tradeoff: some standing warm capacity, in exchange for surviving a burst. Next: the autoscaler signal that actually works.

- A Deployment of pods; a new pod’s timeline shows download then load before the readiness probe passes
- Toggles for a cache PVC (skip the download) and warm min-replicas

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Autoscale On The Right Signal

### Autoscale On The Right Signal

CPU is useless here: vLLM pre-allocates its GPU and KV-cache VRAM, so CPU and GPU memory stay flat under load - a CPU HPA never fires. Scale on the queue: vllm:num_requests_waiting (requests stuck because the GPU is busy) or vllm:gpu_cache_usage_perc. vLLM exports the metric, Prometheus scrapes it, the HPA reads it and adds a GPU.

Here is where most teams get autoscaling wrong. The instinct is to put a **CPU-utilization HPA** on the pods — and it *never scales*, no matter how overloaded the server is. The reason: vLLM **pre-allocates** its GPU compute and KV-cache VRAM at startup, so a pod’s **CPU and GPU-memory stay flat** whether it is idle or slammed. A metric that never moves is a useless autoscaling signal. What *does* move under load is the **queue**: when the GPU is busy, requests pile up waiting, and vLLM exports that as **`vllm:num_requests_waiting`**. That is the signal to scale on — or `vllm:gpu_cache_usage_perc` (the KV cache filling toward preemption). The chain is standard Kubernetes custom metrics: vLLM exports the metric, Prometheus scrapes it, the Kubernetes custom-metrics API surfaces it, and the HPA reads the queue depth and adds a GPU pod (in 2026 GKE can deliver pod metrics to the HPA natively, no adapter). Pair it with the warm min-replicas floor from the last step, so the fleet can absorb a burst while a new pod boots. The block compares the two signals, keyless.

> 🌡️ **Analogy**
>
> It is **a thermostat wired to the wrong sensor**. Imagine a heater whose thermostat, by mistake, reads the room’s *humidity* instead of its temperature. No matter how cold the room gets, the humidity barely changes, so the heater never kicks on — the control loop is broken because it watches a signal that does not respond to what you care about. A CPU HPA on a vLLM pod is exactly that: CPU stays flat under load, so it never triggers. You have to wire the thermostat to the real thermometer — the request queue — which actually rises when the server is overwhelmed.

You set a CPU-utilization HPA on your vLLM pods. Under a heavy load test, the pods are clearly overwhelmed but the HPA never adds a pod. Why?

**📝 `06_autoscale_signal.py`** - *runnable*

In [ ]:
# Autoscale on the RIGHT signal. CPU is useless here: vLLM PRE-ALLOCATES the GPU + KV-cache VRAM at startup,
# so CPU and GPU-memory stay flat no matter the load -> a CPU/memory HPA never scales. Scale on the QUEUE.
CPU_FLAT = 32                    # vLLM sits ~32% CPU whether idle or slammed (VRAM is pre-allocated)
CPU_TARGET = 70                 # a classic CPU HPA target
QUEUE_THRESHOLD = 10            # scale when vllm:num_requests_waiting exceeds this
print("load          CPU%   queue(waiting)   CPU-HPA         queue-HPA (vllm:num_requests_waiting)")
for load, queue in [("light", 0), ("busy", 4), ("overloaded", 38)]:
    cpu_act = "SCALE" if CPU_FLAT > CPU_TARGET else "no-op (never fires)"
    q_act = "SCALE UP" if queue > QUEUE_THRESHOLD else "hold"
    print("  {:<11}   {:>3}       {:>3}           {:<20} {}".format(load, CPU_FLAT, queue, cpu_act, q_act))
print()
print("The CPU HPA is flat at {}% forever, so it NEVER scales - the classic broken LLM autoscaler.".format(CPU_FLAT))
print("Scale on vllm:num_requests_waiting (the queue) or vllm:gpu_cache_usage_perc (the KV cache filling up).")
print("Chain: vLLM exports the metric -> Prometheus -> the k8s custom-metrics API -> HPA (GKE can now do it natively).")

# Output:
# load          CPU%   queue(waiting)   CPU-HPA         queue-HPA (vllm:num_requests_waiting)
#   light          32         0           no-op (never fires)  hold
#   busy           32         4           no-op (never fires)  hold
#   overloaded     32        38           no-op (never fires)  SCALE UP
#
# The CPU HPA is flat at 32% forever, so it NEVER scales - the classic broken LLM autoscaler.
# Scale on vllm:num_requests_waiting (the queue) or vllm:gpu_cache_usage_perc (the KV cache filling up).
# Chain: vLLM exports the metric -> Prometheus -> the k8s custom-metrics API -> HPA (GKE can now do it natively).

- A **CPU HPA** sees a flat CPU no matter the load, because vLLM pre-allocates its VRAM — so it **never scales**.
- The signal that moves is the **queue**: `vllm:num_requests_waiting` rises when the GPU is busy, and a queue HPA scales on it.
- You can also scale on `vllm:gpu_cache_usage_perc` — the KV cache filling toward the preemption you saw in Step 2.
- The chain: vLLM exports the metric, Prometheus scrapes it, the custom-metrics API surfaces it, the HPA adds a GPU pod.

#### 💡 What just happened

⚡ What just happened?A CPU HPA never fires on a vLLM pod because vLLM pre-allocates VRAM, so CPU stays flat; you scale on the queue (vllm:num_requests_waiting) or KV-cache utilization instead. Tradeoff: a custom-metrics pipeline to wire up, in exchange for an autoscaler that actually responds to load. Next: whether running this fleet is worth it at all - the economics.

- A load slider drives a flat CPU line (a CPU HPA that never fires, rose) beside a rising queue-depth line
- The queue line trips the HPA to add a pod (emerald) when it crosses the threshold

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Production Economics

### Production Economics

Cost per million tokens is the GPU price per hour divided by tokens per hour, so vLLM’s throughput is what makes self-serving cheap: the same GPU serves far more tokens at the same hourly cost. Stacking optimizations cuts it another 40-55 percent; the prefix-cache hit rate dominates. It wins at high, steady scale; below that a managed API or Ollama is simpler and cheaper.

Finish with the question that decides whether to run any of this: what does it *cost*? Cost per million tokens is simply the **GPU price per hour divided by the tokens it serves per hour** — and that is why everything in this lesson matters. The GPU costs the same per hour whether it is idle or saturated, so vLLM’s **throughput** is the whole game: continuous batching and PagedAttention let the *same* GPU serve roughly ten to twenty times more tokens per hour, which makes each token roughly **an order of magnitude cheaper** than naive serving. Stacking further optimizations — quantization, chunked prefill, and a good **prefix-cache hit rate** (which dominates, because the cost shape is prefill-heavy) — cuts cost per million tokens another **forty to fifty-five percent**. For reliability you run multiple replicas with rolling updates and a disruption budget so a node loss does not drop the service. And the honest close, straight from 13.5: vLLM on GKE wins at **high, steady scale**, where a busy GPU is genuinely cheap per token; below that, a managed API or Ollama is simpler and cheaper, because a self-hosted fleet only pays off when the GPUs stay full. The block prices it, keyless.

> 🏭 **Analogy**
>
> A production GPU is **a factory machine**. It costs the same to keep running for an hour whether it stamps out one part or a thousand — the electricity, the lease, the operator are fixed. So the cost *per part* is all about **keeping it busy**: a machine running flat out makes cheap parts, a machine idling in the corner makes ruinously expensive ones. vLLM is the retooling that lets the same machine stamp ten times as many parts per hour, so each part costs a fraction as much — but only if you actually feed it enough work to run full. An idle high-end GPU is the most expensive way to make a token there is.

vLLM serves the same GPU 12 times more tokens per hour than naive serving. The GPU costs the same per hour either way. What happens to the cost per token?

**📝 `07_production_economics.py`** - *runnable*

In [ ]:
# Cost per 1M tokens = GPU price/hour divided by tokens/hour. vLLM's throughput is what makes self-serving
# cheap: the same GPU that costs the same per hour serves far more tokens, so each token costs far less.
GPU_HR = 3.50                    # illustrative H100 $/hr
for name, tps in [("naive static serving", 200), ("vLLM (continuous + paged)", 2400)]:
    tok_per_hr = tps * 3600
    per_1m = GPU_HR / tok_per_hr * 1_000_000
    print("  {:<28} {:>4} tok/s -> ${:.2f} per 1M tokens".format(name, tps, per_1m))
print()
naive = GPU_HR / (200 * 3600) * 1e6
vllm = GPU_HR / (2400 * 3600) * 1e6
print("Same GPU, same ${:.2f}/hr: vLLM is {:.0f}x cheaper per token purely from higher throughput.".format(GPU_HR, naive / vllm))
print("Stacking optimisations (quantization, chunked prefill, a good prefix-cache hit rate) cuts $/1M another 40-55%.")
print("The decision: vLLM on GKE wins at HIGH, STEADY scale (a busy GPU is cheap per token); below that, a managed")
print("API or Ollama (Lesson 13.5) is simpler and cheaper - self-serving only pays off when the GPUs stay full.")

# Output:
#   naive static serving          200 tok/s -> $4.86 per 1M tokens
#   vLLM (continuous + paged)    2400 tok/s -> $0.41 per 1M tokens
#
# Same GPU, same $3.50/hr: vLLM is 12x cheaper per token purely from higher throughput.
# Stacking optimisations (quantization, chunked prefill, a good prefix-cache hit rate) cuts $/1M another 40-55%.
# The decision: vLLM on GKE wins at HIGH, STEADY scale (a busy GPU is cheap per token); below that, a managed
# API or Ollama (Lesson 13.5) is simpler and cheaper - self-serving only pays off when the GPUs stay full.

**📝 `vllm-gke.sh`** - *illustrative*

In [ ]:
# PRODUCTION vLLM ON GKE - an illustrative minimal example (a GPU node pool, the Deployment, a queue HPA).
# 1) A GPU node pool with the MODERN driver flag (no manual DaemonSet), autoscaling the nodes:
gcloud container node-pools create gpu-pool --cluster prod --machine-type a3-highgpu-1g \
  --accelerator type=nvidia-h100-80gb,count=1,gpu-driver-version=default \
  --enable-autoscaling --min-nodes 1 --max-nodes 8

# 2) The vLLM Deployment - one GPU per pod, OpenAI-compatible on :8000, weights cached on a PVC:
#    image: vllm/vllm-openai:v0.11.0                 # PIN a version, never :latest
#    args: --model meta-llama/Llama-3.3-70B-Instruct-FP8
#          --tensor-parallel-size 1 --gpu-memory-utilization 0.90 --max-model-len 8192 --max-num-seqs 256
#    resources: { limits: { nvidia.com/gpu: 1 } }
#    readinessProbe: { httpGet: { path: /health, port: 8000 }, initialDelaySeconds: 300 }  # cold start

# 3) Autoscale on the QUEUE, not CPU (vLLM pre-allocates VRAM, so CPU never moves):
#    HorizontalPodAutoscaler -> metric vllm:num_requests_waiting, averageValue 10, minReplicas 2

# 4) Your app calls it UNCHANGED - the same OpenAI client, pointed at the in-cluster service:
#    client = OpenAI(base_url="http://vllm-service/v1", api_key="cluster")
# Output: (illustrative minimal example - needs a GKE cluster + GPU quota + the model pulled; not run here.)

- Cost per token is the **GPU price per hour over tokens per hour**, so vLLM’s throughput makes the same GPU roughly an order of magnitude cheaper per token.
- Stacking quantization, chunked prefill, and a good **prefix-cache hit rate** cuts cost per million tokens another forty to fifty-five percent.
- The **illustrative GKE artifact** ties the stack together: a GPU node pool, a version-pinned vLLM Deployment, and an HPA on the request queue.
- The decision: self-serving wins at **high, steady utilization** (a full GPU is cheap per token); below that, a managed API or Ollama is simpler and cheaper.

#### 💡 What just happened

⚡ What just happened?Cost per token is GPU $/hr over tokens/hr, so vLLM’s throughput makes each token roughly an order of magnitude cheaper - and stacked optimizations plus a good prefix-cache hit rate cut it another 40-55 percent. Tradeoff: the ops of a GPU fleet, worth it only at high steady utilization. That is the whole lesson: keep the GPU full, or rent an API.

- A throughput slider moves the cost-per-1M-token curve down as the GPU serves more
- A naive-vs-vLLM marker and a managed-API line show when self-serving wins

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> Production serving is two engines and a fleet. It starts with **continuous batching**, which refills a finished GPU slot from the queue every forward pass so the GPU never idles on padding (Step 1), fed by **PagedAttention**, which packs the KV cache into fixed pages so far more sequences fit (Step 2). You set the fullness with the **max-num-seqs knob**, trading per-request latency for aggregate throughput at the knee (Step 3), and serve a too-big model by sharding it across GPUs with **tensor parallelism** behind one OpenAI-compatible endpoint (Step 4). GKE runs it as a **Deployment** of GPU pods, designed around the minutes-long **cold start** (Step 5), and autoscaled on the **request queue** — never CPU, which is flat (Step 6). And the **economics** close it: throughput makes a busy GPU cheap per token, so self-serving wins only when the fleet stays full (Step 7). Ask two questions of any production stack: does continuous batching keep the GPU full, and can you keep the fleet busy enough to beat a managed API?

| Option | Concurrency | Best for |
|---|---|---|
| Ollama (13.5) | one at a time | dev, a laptop, low-traffic tools |
| vLLM on GKE | hundreds per GPU, autoscaled | production self-hosting at high, steady scale |
| Managed API | unlimited (someone else’s fleet) | low or spiky volume, no ops capacity, frontier quality |

> 📦 **Info**
>
> ➡️ Where this goes nextThat completes the cost-and-performance module: you can price a model, cut its tokens, route it, speed it up, self-host it, and now serve it at production scale. Running that serving layer as an operational discipline — instrumentation, prompt versioning, evaluation, and error analysis — comes next, in Module 14 (LLMOps), where Lesson 14.5 covers serving and cost optimization directly. Scraping vLLM’s queue-depth and KV-cache metrics into your dashboards is the observability of Lesson 12.8, and the rolling-update and disruption-budget patterns build on the reliability of Lesson 12.2. The primary references are the [PagedAttention paper](https://arxiv.org/abs/2309.06180), the [vLLM docs](https://docs.vllm.ai/), [continuous batching](https://www.anyscale.com/blog/continuous-batching-llm-inference), and [vLLM on GKE](https://github.com/GoogleCloudPlatform/accelerated-platforms/blob/main/docs/use-cases/inferencing/README.md).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **vLLM& GKE Production**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-13_6.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-13.6.html` - regenerate this notebook whenever the source HTML is updated.*
